In [99]:
# to configure the llm to load

import os
import dotenv

dotenv.load_dotenv("../.env")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ")


# model = "openai/gpt-oss-120b"
# model = "openai/gpt-oss-20b"
model = "llama-3.3-70b-versatile"
# model = "meta-llama/llama-4-scout-17b-16e-instruct"

In [100]:
# to load the model from Groq using langchain
# init_chat_model()

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model=model,
    model_provider="groq",
    temperature=0
)

# to check the model
response = llm.invoke("What is the capital of France?")
print(response)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.011194721, 'completion_tokens_details': None, 'prompt_time': 0.001943621, 'prompt_tokens_details': None, 'queue_time': 0.098442171, 'total_time': 0.013138342}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2f22-6365-7642-a2e1-cc338db40bfe-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


In [101]:
# tools for agents
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchResults
from langchain_community.utilities import WikipediaAPIWrapper

# a tool to search web
tool_search = DuckDuckGoSearchResults()

# a tool to query wikipedia
wiki_api = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=10000
)
tool_wiki = WikipediaQueryRun(api_wrapper=wiki_api)

tool_set = [tool_search, tool_wiki]


In [102]:
from pprint import pprint
llm_with_tools = llm.bind_tools(tool_set)

response = llm_with_tools.invoke("Who is the president of the United States?")
pprint(response)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'amwkgeq52', 'function': {'arguments': '{"query":"current president of the United States"}', 'name': 'duckduckgo_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 395, 'total_tokens': 418, 'completion_time': 0.039322507, 'completion_tokens_details': None, 'prompt_time': 0.021400728, 'prompt_tokens_details': None, 'queue_time': 0.092877877, 'total_time': 0.060723235}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2f22-7dbf-7121-ab3b-b090f356da78-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'current president of the United States'}, 'id': 'amwkgeq52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 395, 'output_tokens': 23, 'total_tokens': 418})


In [103]:
pprint(response.tool_calls)
# for tool in tool_set:
#     print(tool.name)

[{'args': {'query': 'current president of the United States'},
  'id': 'amwkgeq52',
  'name': 'duckduckgo_results_json',
  'type': 'tool_call'}]


In [105]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
# to bind the tools to the llm
llm_with_tools = llm.bind_tools(tool_set)

# now need to define manual tool calling
tool_mapping = {
    "duckduckgo_results_json": tool_search,
    "wikipedia": tool_wiki
}

chat_history = []

# system prompt to guide the llm to use the tools effectively
RESEARCH_SYSTEM_PROMPT = """
You are an expert researcher. Use tools multiple times to verify facts.
Identify gaps in information and perform follow-up searches.
"""

chat_history.append(SystemMessage(content=RESEARCH_SYSTEM_PROMPT))  

user_query = """
why nvidia is not seeing the GPU development by AMD, apple and intel as a threat, however the 
A15 and A16 chips from Tesla as a serious threat?
"""

# user_query = """
# why the statement by the Pakistan military chief Asif Munir, Jinna was a Shia, is controversial?
# """


chat_history.append(HumanMessage(content=user_query))

def call_tool(response):
    calls = response.tool_calls
    tool_messages = []

    for call in calls:
        # print(call.get("name"))
        tool_name = call.get("name")
        args = call.get("args")

        tool = tool_mapping.get(tool_name)
        if tool:
            observation = tool.invoke(args)
            tool_messages.append(
                ToolMessage(
                    content=str(observation), 
                    tool_call_id=call.get("id")
                )
            )

        else:
            tool_messages.append(
                ToolMessage(
                    content=f"Tool '{tool_name}' not found in tool mapping.", 
                    tool_call_id=response.call.get("id")
                )
            )  
    return tool_messages
    
while True:
    response = llm_with_tools.invoke(chat_history)
    # need to append the reponse to the chat_history to maintain the context for the llm
    chat_history.append(response)

    if response.tool_calls:
        tool_response = call_tool(response)
        pprint(response.tool_calls)
        chat_history.extend(
            # append the tool response with the id to link it to the tool call in the response
            tool_response
        )
    else:
        pprint(response.content)
        break 

[{'args': {'query': 'NVIDIA views on AMD, Apple, Intel GPU development'},
  'id': 'zbm6h5efp',
  'name': 'duckduckgo_results_json',
  'type': 'tool_call'},
 {'args': {'query': 'NVIDIA views on Tesla A15 and A16 chips as a threat'},
  'id': 'hnkm08kap',
  'name': 'duckduckgo_results_json',
  'type': 'tool_call'},
 {'args': {'query': 'NVIDIA'},
  'id': 'yv5jv6p7j',
  'name': 'wikipedia',
  'type': 'tool_call'},
 {'args': {'query': 'Tesla A15 and A16 chips'},
  'id': '14swj347b',
  'name': 'wikipedia',
  'type': 'tool_call'}]
('Nvidia is not seeing the GPU development by AMD, Apple, and Intel as a '
 'threat because they have a strong market share and a wide range of products '
 'that cater to different markets, including gaming, professional '
 'visualization, and artificial intelligence. Additionally, Nvidia has been '
 'investing heavily in research and development, which has enabled them to '
 'stay ahead of the competition.\n'
 '\n'
 'On the other hand, Nvidia views the A15 and A16 c

In [106]:
pprint(response.content)

('Nvidia is not seeing the GPU development by AMD, Apple, and Intel as a '
 'threat because they have a strong market share and a wide range of products '
 'that cater to different markets, including gaming, professional '
 'visualization, and artificial intelligence. Additionally, Nvidia has been '
 'investing heavily in research and development, which has enabled them to '
 'stay ahead of the competition.\n'
 '\n'
 'On the other hand, Nvidia views the A15 and A16 chips from Tesla as a '
 'serious threat because they are designed for autonomous driving and '
 'artificial intelligence applications, which are areas where Nvidia is also '
 "actively involved. Tesla's chips are designed to be highly efficient and "
 "powerful, which could potentially disrupt Nvidia's dominance in the market.\n"
 '\n'
 "It's worth noting that Nvidia has been expanding its product lines to "
 'include more specialized chips for specific applications, such as the Tegra '
 'line of mobile processors for smart

## Observations
- openai-oss models
    - resoning champion
    - one function call in an AIMessage
- llama-3-versetile
    - multiple function call in an AI message

### System message
- a system message can drastically impove the output quality
    - give personalization to the llm
    - ask explicitly to counter verify the information gathered from open source

### different model strentghts
|Model Family|	Primary Strength|	Tool Calling Style|	Best Use Case|	Architecture|
|---|---|---|---|---|
|Llama 4 / 3.3|	Speed & Ubiquity|	Parallel (Aggressive/Fast)|	General Assistants & Rapid API Orchestration|	Dense & MoE (Maverick)|
|Mistral (Dense)|	Efficiency & JSON Precision|	Precise (Multi-tool)|	RAG, Edge Deployments & Structured Data|	Optimized Dense
|Mixtral (MoE)|	Technical Depth & Coding|	Reliable (Mixed)|	Multilingual Support & Scientific Research|	Sparse MoE (8x22B)
|GPT-OSS|	Logic & Self-Correction|	Sequential (Deep Reasoning)|	Complex Multi-step Agentic Workflows|	Large-Scale Reasoning|